In [12]:
import numpy as np 
import pandas as pd

In [13]:
df = pd.read_csv("powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [14]:
from sklearn.model_selection import train_test_split
X = df.drop("PE" , axis = 1)
y = df["PE"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)


In [15]:
X.head()

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20


In [16]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
X_train_scaled 

array([[ 0.01845156,  0.84137636,  0.49926379,  0.09294124],
       [-1.29179451, -0.98482991,  3.35699973, -0.36102302],
       [-0.75322711, -1.2134986 ,  0.43846089, -0.3004026 ],
       ...,
       [-0.21867886, -0.83709894,  0.36076831, -0.84874182],
       [ 0.95089661,  1.13840991, -0.42460239, -0.46641988],
       [-1.77275395, -1.19228191,  1.91462001,  0.91131686]],
      shape=(6410, 4))

In [18]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled , dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled , dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)

In [19]:
type(X_train_scaled)

numpy.ndarray

In [20]:
type(X_test_scaled)

numpy.ndarray

In [25]:
from torch.utils.data import TensorDataset , DataLoader
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [26]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32 )

## DEEP LEARNING

In [34]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN , self).__init__()

        self.model = nn.Sequential(
               #1st hidden layer
               nn.Linear(X_train.shape[1], 6 ),
               nn.ReLU(),
    
               #2nd hidden layer
    
               nn.Linear(6, 6 ),
               nn.ReLU(),
    
               #output
               nn.Linear(6, 1 )
    
            
        )
    def forward(self , x):
        return self.model(x)


In [37]:
import torch.optim as optim

model = ANN()
# loss , optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 100
train_losses= []

for epoch in range(epochs):
    model .train()
    loss_runner = 0.0
    for xb , yb in train_loader:
        #xb is feature 1 and labels of batch is yb 

        outputs = model(xb) # forward prob
        loss = crietrion(outputs)
        loss.backward()
        optimizer.step()#update params

        loss_runner += loss.item() # loss is tensor ==> py float
        
    epouch_train_loss = loss_runner/len(train_loader)
    train_losses.append(epouch_train_loss)